# w2 · Secuencias y LSTM PyTorch · Ejercicios

12 ejercicios de ⭐ a ⭐⭐⭐, con prioridad en Polars, secuencias y tensores. Edita [`src/estudio/ejercicios.py`](src/estudio/ejercicios.py). Los `TODO`, contratos de shapes, pistas y `assert` quedan en `src`; el notebook importa funciones, muestra resultados y llama a verificaciones. La recarga automática detecta los cambios del `.py`.

El flujo es **logs → días → listas por usuario → calendario → tensor → normalización → Dataset/DataLoader → LSTM → entrenamiento → Optuna**. Tras cada ejercicio hay una celda que comprueba shapes, dtypes y valores con fixtures pequeñas e independientes de los datos grandes. Las comprobaciones de todo el pipeline requieren resolver los ejercicios anteriores.

Por defecto: **192 usuarios por cohorte**, 90 días, 20 canales, CPU, **3 épocas y 3 trials**. `MODO="mini"` genera datos sintéticos para practicar sin descargar KKBOX. Sus métricas no son resultados reales. `MODO="real"` lee las mismas fuentes raw de `workshops/data/` y mantiene el límite de usuarios. Usa las dependencias del proyecto: Polars, NumPy, pandas, PyTorch, Optuna, scikit-learn, matplotlib e IPython/Jupyter.

Se construyen los **8 canales de escucha** como ejercicios; los **12 de transacciones** reutilizan el constructor original, sin cambios. Se mantiene la rejilla diaria: un cero de padding representa un día sin registro, no un evento eliminado. Se normalizan solo los siete canales numéricos de escucha; los canales de transacciones conservan su encoding original log1p/signed-log1p/flags. El clipping y escalado completo por dominios permanecen en el pipeline original.

Febrero: cutoff 2017-01-31, split estratificado por usuario 75% fit / 12.5% stop / 12.5% tune. Marzo: cutoff 2017-02-28, test final. Usuarios recurrentes pueden aparecer en ambos meses con distintos snapshots. Scaler y pesos se aprenden en fit; early stopping consulta stop; Optuna y umbral consultan tune. Marzo no interviene en esas decisiones. El refit global, AMP y exportación avanzada se conservan en los notebooks originales.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import torch

W2_DIR = next(
    p for p in (Path.cwd(), Path.cwd() / "workshops" / "w2")
    if (p / "w2_sequence_preprocessing.ipynb").is_file()
)
sys.path.insert(0, str(W2_DIR.parent))
from w2.src.estudio import ejercicios as practica

from w2.src.estudio import verificaciones as checks
from w2.src.estudio.soporte import (
    CUTOFFS, cargar_datos, combinar_canales, dividir_usuarios,
    fijar_semilla, perdida_probabilidades, ejecutar_estudio, evaluar_mejor, graficar,
)

MODO = "mini"
N_USERS = 192
BATCH_SIZE = 16
EPOCHS = 3
N_TRIALS = 3
CUTOFF = CUTOFFS["train_feb"]


In [ ]:
users, logs_raw, transactions = cargar_datos(W2_DIR, modo=MODO, n_users=N_USERS)


## 01 · ⭐ · Ordenar logs y recortar la ventana causal

Selecciona solo los usuarios de la cohorte y los últimos 90 días hasta el cutoff inclusive. Añade el offset calendario `day` y ordena por posición de usuario y fecha. No agregues todavía.

Edita `ordenar_y_filtrar` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Por qué ordenar por uid no garantiza el orden correcto del eje batch si existe row?

In [ ]:
windowed = practica.ordenar_y_filtrar(logs_raw, users["train_feb"], CUTOFF)
print(windowed.shape)
windowed.head(3)


In [ ]:
checks.verificar_01(practica.ordenar_y_filtrar)


## 02 · ⭐ · Consolidar registros por usuario y día

Puede haber varios registros para el mismo usuario y día. Suma los siete contadores antes de aplicar `log1p`; conserva una única fila por usuario-día.

Edita `agregar_dias` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Por qué log1p debe ir después de sumar duplicados diarios?

In [ ]:
daily = practica.agregar_dias(windowed)
print(daily.shape)
daily.head(3)


In [ ]:
checks.verificar_02(practica.agregar_dias)


## 03 · ⭐⭐ · Construir listas temporales con group_by y over

Ordena antes de agrupar. Usa `over` para numerar los días observados dentro de cada usuario y construye listas con `group_by/agg`. Conserva también a los usuarios sin logs, con listas vacías. Los contadores de escucha se transforman con `log1p`.

Edita `agrupar_secuencias` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Qué diferencia hay entre el índice de observación steps y el offset calendario days?

In [ ]:
sequences = practica.agrupar_secuencias(daily, users["train_feb"])
print(sequences.shape)
sequences.head(3)


In [ ]:
checks.verificar_03(practica.agrupar_secuencias)


## 04 · ⭐⭐ · Padding y truncado sin comprimir el calendario

Construye una rejilla calendario `(N,90,8)`. Escribe cada observación en su offset real; rellena días ausentes con cero y marca días observados con `active=1`. Descarta offsets fuera de la ventana. Un día observado sin canciones debe distinguirse de un día sin registro.

Edita `padding_y_truncado` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Por qué un LSTM de días calendario no debe recibir solo los días activos pegados entre sí?

In [ ]:
listening = practica.padding_y_truncado(sequences)
print("Escucha:", listening.shape, listening.dtype)


In [ ]:
checks.verificar_04(practica.padding_y_truncado)


## 05 · ⭐⭐ · Convertir a tensores batch × tiempo × features

Convierte el array a un tensor contiguo `float32` con ejes `(usuario,tiempo,features)` y las etiquetas a `(N,1)`. Los 12 canales de transacciones se añaden con el constructor original, ya preparado, antes de llamar a tu función.

Edita `convertir_tensores` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Qué ocurriría si usaras (N,F,T) en un LSTM configurado con batch_first=True?

In [ ]:
combined = combinar_canales(listening, transactions, users["train_feb"], CUTOFF)
y_raw = users["train_feb"].sort("row")["is_churn"].to_numpy()
X, y = practica.convertir_tensores(combined, y_raw)
print("X:", tuple(X.shape), X.dtype, "| y:", tuple(y.shape), y.dtype)


In [ ]:
checks.verificar_05(practica.convertir_tensores)


## 06 · ⭐⭐ · Normalizar solo con entrenamiento y días observados

Aprende media y desviación poblacional de los siete canales de escucha usando solo `fit` y solo días con `active=1`. Ignora padding al aprender y conserva sus ceros al transformar. Mantén flags y canales de transacciones intactos; no uses stop, tune ni marzo para aprender estadísticas.

Edita `normalizar_secuencias` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Qué sesgo introducen los días de padding en la media? ¿Qué fuga introduce usar toda la cohorte?

In [ ]:
split = dividir_usuarios(users["train_feb"])
X_scaled, scaler_state = practica.normalizar_secuencias(X, split["fit"])
print("Split:", {name: len(rows) for name, rows in split.items()})
print("Media/std:", scaler_state["mean"].shape, scaler_state["std"].shape)


In [ ]:
checks.verificar_06(practica.normalizar_secuencias)


## 07 · ⭐⭐⭐ · Implementar Dataset y DataLoader

Completa `ChurnDataset.__len__`, `__getitem__` y `crear_loaders`. El índice local de un Dataset debe resolver primero el índice global del usuario. Solo fit se baraja y descarta el último batch incompleto por BatchNorm; stop/tune conservan todos los usuarios en orden.

Edita `ChurnDataset + crear_loaders` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Por qué __getitem__(i) no puede tomar simplemente X[i] cuando rows es un subconjunto?

In [ ]:
loaders = practica.crear_loaders(X_scaled, y, split, batch_size=BATCH_SIZE)
xb, yb = next(iter(loaders["fit"]))
print("Batch:", tuple(xb.shape), xb.dtype, "|", tuple(yb.shape), yb.dtype)


In [ ]:
checks.verificar_07(practica.ChurnDataset, practica.crear_loaders)


## 08 · ⭐⭐⭐ · Escribir la arquitectura y forward del LSTM

Implementa LSTM con `batch_first=True`, BatchNorm del último estado y una capa lineal de salida. Devuelve logits y los estados ocultos. El sigmoid pertenece a inferencia; la pérdida de entrenamiento recibe logits.

Edita `LSTMChurn` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Por qué sigmoid en forward junto a BCEWithLogitsLoss sería un error?

In [ ]:
fijar_semilla()
example_model = practica.LSTMChurn(input_dim=X.shape[2], hidden_dim=16, num_layers=1)
logits, hn, cn = example_model(xb)
print("Logits:", tuple(logits.shape), "| hn:", tuple(hn.shape), "| cn:", tuple(cn.shape))


In [ ]:
checks.verificar_08(practica.LSTMChurn)


## 09 · ⭐⭐⭐ · Escribir una época de entrenamiento

Activa modo entrenamiento, limpia gradientes por batch, calcula BCE con logits, propaga gradientes, recorta la norma y actualiza el optimizador. Devuelve una pérdida ponderada por número de usuarios, no la media simple de las pérdidas de cada batch.

Edita `entrenar_epoca` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Qué pasa si olvidas zero_grad entre dos batches?

In [ ]:
optimizer = torch.optim.AdamW(example_model.parameters(), lr=1e-3)
train_loss = practica.entrenar_epoca(example_model, loaders["fit"], optimizer)
print("BCE de entrenamiento:", round(train_loss, 5))


In [ ]:
checks.verificar_09(practica.entrenar_epoca)


## 10 · ⭐⭐⭐ · Evaluar sin gradientes ni actualización del modelo

Desactiva entrenamiento y gradientes, calcula probabilidades y concatena todos los batches. Incluye el último batch pequeño. El estado del modelo y los buffers de BatchNorm deben permanecer iguales.

Edita `predecir` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Qué cambia al usar no_grad sin llamar a eval cuando hay BatchNorm?

In [ ]:
p_stop = practica.predecir(example_model, loaders["stop"])
print("Probabilidades stop:", p_stop.shape, p_stop.dtype)
print("Log loss stop:", round(perdida_probabilidades(loaders["stop"], p_stop), 5))


In [ ]:
checks.verificar_10(practica.predecir)


## 11 · ⭐⭐⭐ · Early stopping y restauración del mejor estado

Combina las funciones anteriores: entrena en fit, observa la pérdida de stop, cuenta épocas sin mejora y restaura una copia del mejor estado. Tune se reserva para el siguiente ejercicio.

Edita `entrenar_con_early_stopping` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Por qué state_dict sin deepcopy puede dejar de representar la mejor época?

In [ ]:
fijar_semilla()
model = practica.LSTMChurn(input_dim=X.shape[2], hidden_dim=16, num_layers=1)
training = practica.entrenar_con_early_stopping(model, loaders, epochs=EPOCHS)
print("Mejor época:", training["best_epoch"], "| stop loss:", round(training["best_stop_loss"], 5))


In [ ]:
checks.verificar_11(practica.entrenar_con_early_stopping)


## 12 · ⭐⭐⭐ · Escribir el objective de Optuna

Sugiere hiperparámetros pequeños, ajusta el scaler en fit, crea loaders y modelo, entrena con early stopping en stop y devuelve log loss en tune. La creación del estudio y las métricas ya están escritas. Marzo solo se usa después de seleccionar la configuración.

Edita `objective` en [`ejercicios.py`](src/estudio/ejercicios.py). Los detalles de entrada/salida y las pistas están junto al hueco.

Para razonar: ¿Qué papel distinto cumplen stop, tune y test? ¿Por qué el objetivo devuelve una pérdida y no accuracy?

In [ ]:
study = ejecutar_estudio(practica, X, y, split, n_trials=N_TRIALS, epochs=EPOCHS, batch_size=BATCH_SIZE)
print("Mejores parámetros:", study.best_params)
print("Log loss tune:", round(study.best_value, 5))


In [ ]:
checks.verificar_12(practica.objective)


## Evaluación final y plots · ya preparados

Se reproduce la configuración ganadora usando fit/stop y se fija el umbral en tune. Se construye marzo con sus propios 90 días y se transforma con el scaler de fit, sin reajustarlo. Se reutilizan las métricas y plots originales. El modelo conserva el entrenamiento en fit; el refit sobre todo febrero queda en el notebook original.

Para terminar: reinicia el kernel y ejecuta todo. Para usar datos reales, cambia `MODO` y vuelve a ejecutar desde la carga.

In [ ]:
metrics, y_test, p_test = evaluar_mejor(
    practica, study, X, y, split, users, logs_raw, transactions,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
)
metrics[["part", "log_loss", "roc_auc", "average_precision", "f1_churn", "threshold"]].round(5)


In [ ]:
figures_dir = graficar(y_test, p_test, study, W2_DIR, MODO)
print("Figuras:", figures_dir)
